# Bloque 6 — Audio e IA
## Notebook 00: Transcripción local de audio con Whisper

---

### ¿Qué es Whisper?

**Whisper** es un modelo de reconocimiento de voz (speech-to-text) desarrollado por OpenAI y publicado como open-source. Funciona igual que Ollama pero para audio: lo ejecutamos en local, sin API, sin coste por petición, sin límite de uso.

Técnicamente es un **encoder-decoder seq2seq** basado en transformers, entrenado con 680.000 horas de audio en 99 idiomas. El encoder convierte el audio en una representación vectorial; el decoder genera el texto token a token, igual que un LLM de texto.

```
[archivo de audio] → Encoder (espectrograma → vectores) → Decoder (vectores → texto)
```

### ¿Por qué `faster-whisper` y no `openai-whisper`?

Mismo modelo, implementación más eficiente (CTranslate2 en vez de PyTorch). Resultado idéntico, pero:
- 4× más rápido en CPU
- Menor uso de memoria
- Cuantización `int8` nativa (mismo truco que con Ollama)

### Modelos disponibles

| Modelo | Tamaño | Velocidad CPU | Cuándo usarlo |
|---|---|---|---|
| `tiny` | 39 MB | muy rápido | tests, demos rápidas |
| `base` | 74 MB | rápido | demos ágiles |
| `small` | 244 MB | moderado | **demo en clase — buen equilibrio** |
| `medium` | 769 MB | lento en CPU | producción con GPU |
| `large-v3` | 1.5 GB | muy lento en CPU | máxima calidad |

Los pesos se descargan automáticamente de HuggingFace la primera vez y se cachean en `~/.cache/huggingface/`.

In [1]:
# ─── VERIFICACIÓN DEL ENTORNO ─────────────────────────────────────────────────
# Comprobamos que faster-whisper y ffmpeg están disponibles.
# Si ffmpeg falla, instálalo antes de continuar:
#   sudo apt install -y ffmpeg

import importlib
import shutil

# faster-whisper
try:
    import faster_whisper
    print(f'✓ faster-whisper {faster_whisper.__version__}')
except ImportError:
    print('✗ faster-whisper NO encontrado — ejecuta: uv sync')

# gtts (para generar el audio de demo)
try:
    import gtts
    print(f'✓ gTTS {gtts.__version__}')
except ImportError:
    print('✗ gTTS NO encontrado — ejecuta: uv sync')

# ffmpeg (necesario para decodificar mp3/m4a/etc.)
if shutil.which('ffmpeg'):
    print('✓ ffmpeg disponible')
else:
    print('✗ ffmpeg NO encontrado — instala con: sudo apt install -y ffmpeg')

✓ faster-whisper 1.2.1
✓ gTTS 2.5.4
✓ ffmpeg disponible


## Generar el audio de demo

Usamos `gTTS` (Google Text-to-Speech) para generar un clip de audio en español a partir de un texto. Esto hace el notebook completamente autocontenido — no hace falta traer ningún archivo de audio.

El texto simula una nota informativa sobre un incidente de ransomware, conectando con la temática del curso.

> **Alternativa**: si quieres usar tu propio audio, sustituye la variable `AUDIO_PATH` por la ruta a tu fichero `.mp3` o `.wav` y salta la celda de generación.

In [2]:
# ─── GENERAR AUDIO DE DEMO CON gTTS ──────────────────────────────────────────
# gTTS convierte texto a audio MP3 usando la API de Google TTS.
# Necesita conexión a internet SOLO en esta celda; el resto del notebook es 100% local.

from pathlib import Path
from gtts import gTTS

AUDIO_PATH = Path('audio/demo.mp3')
AUDIO_PATH.parent.mkdir(exist_ok=True)

TEXTO_DEMO = (
    'En mayo de 2021, el grupo de ransomware DarkSide atacó el oleoducto Colonial Pipeline '
    'en Estados Unidos, provocando el cierre temporal de uno de los principales sistemas de '
    'distribución de combustible del país. '
    'El ataque comenzó con credenciales de acceso robadas a través de una cuenta VPN sin '
    'autenticación de doble factor. '
    'En menos de dos horas, los atacantes cifraron más de cien gigabytes de datos y '
    'exigieron un rescate de cuatro coma cuatro millones de dólares en Bitcoin. '
    'Este incidente marcó un punto de inflexión en la percepción del ransomware como '
    'amenaza a infraestructuras críticas nacionales.'
)

if not AUDIO_PATH.exists():
    tts = gTTS(text=TEXTO_DEMO, lang='es', slow=False)
    tts.save(AUDIO_PATH)
    print(f'Audio generado: {AUDIO_PATH} ({AUDIO_PATH.stat().st_size / 1024:.1f} KB)')
else:
    print(f'Audio ya existe: {AUDIO_PATH} ({AUDIO_PATH.stat().st_size / 1024:.1f} KB)')

print(f'\nTexto original ({len(TEXTO_DEMO.split())} palabras):')
print(TEXTO_DEMO)

Audio ya existe: audio/demo.mp3 (390.2 KB)

Texto original (97 palabras):
En mayo de 2021, el grupo de ransomware DarkSide atacó el oleoducto Colonial Pipeline en Estados Unidos, provocando el cierre temporal de uno de los principales sistemas de distribución de combustible del país. El ataque comenzó con credenciales de acceso robadas a través de una cuenta VPN sin autenticación de doble factor. En menos de dos horas, los atacantes cifraron más de cien gigabytes de datos y exigieron un rescate de cuatro coma cuatro millones de dólares en Bitcoin. Este incidente marcó un punto de inflexión en la percepción del ransomware como amenaza a infraestructuras críticas nacionales.


## Transcribir con Whisper

Cargamos el modelo `small` y transcribimos el audio. La primera vez tarda unos segundos extra para descargar los pesos (~244 MB) de HuggingFace.

In [3]:
# ─── CARGAR EL MODELO WHISPER ─────────────────────────────────────────────────
import time
from faster_whisper import WhisperModel

# device='cpu' → funciona en cualquier máquina sin GPU
# compute_type='int8' → cuantización de 8 bits: más rápido, mismo resultado
# Si tienes GPU NVIDIA: device='cuda', compute_type='float16'
print('Cargando modelo small (descarga ~244 MB si es la primera vez)...')
t0 = time.time()
model = WhisperModel('small', device='cpu', compute_type='int8')
print(f'Modelo listo en {time.time() - t0:.1f}s')

Cargando modelo small (descarga ~244 MB si es la primera vez)...
Modelo listo en 0.6s


In [4]:
# ─── TRANSCRIBIR ──────────────────────────────────────────────────────────────
# model.transcribe() devuelve:
#   - segments: generador de segmentos con texto, timestamps y probabilidades
#   - info: metadatos (idioma detectado, duración del audio, etc.)
#
# language='es' fuerza el español. Si lo omites, Whisper detecta el idioma automáticamente.

t0 = time.time()
segments, info = model.transcribe(str(AUDIO_PATH), language='es')

# Consumimos el generador y guardamos los segmentos en una lista.
# (El generador es lazy: sin esto, Whisper no procesa nada todavía.)
lista_segmentos = list(segments)
tiempo_transcripcion = time.time() - t0

print(f'Idioma detectado : {info.language} (confianza: {info.language_probability:.0%})')
print(f'Duración del audio: {info.duration:.1f}s')
print(f'Tiempo de transcripción: {tiempo_transcripcion:.1f}s')
print(f'Factor de velocidad: {info.duration / tiempo_transcripcion:.1f}× tiempo real')
print(f'Segmentos: {len(lista_segmentos)}')

Idioma detectado : es (confianza: 100%)
Duración del audio: 49.9s
Tiempo de transcripción: 2.8s
Factor de velocidad: 18.1× tiempo real
Segmentos: 9


In [5]:
# ─── VER LOS SEGMENTOS CON TIMESTAMPS ────────────────────────────────────────
# Cada segmento tiene: start, end, text, words (si beam_size > 1)
# El formato [MM:SS.d → MM:SS.d] es el estándar en transcripción profesional.

def fmt_tiempo(segundos: float) -> str:
    m, s = divmod(segundos, 60)
    return f'{int(m):02d}:{s:04.1f}'


print('TRANSCRIPCIÓN CON TIMESTAMPS:\n')
for seg in lista_segmentos:
    print(f'[{fmt_tiempo(seg.start)} → {fmt_tiempo(seg.end)}]  {seg.text.strip()}')

# Texto plano completo (sin timestamps)
transcripcion_completa = ' '.join(seg.text.strip() for seg in lista_segmentos)
print(f'\nTEXTO COMPLETO ({len(transcripcion_completa.split())} palabras):')
print(transcripcion_completa)

TRANSCRIPCIÓN CON TIMESTAMPS:

[00:00.0 → 00:08.4]  En mayo de 2021, el grupo de ransomware darksite atacó el oleoducto colonial pipeline en Estados
[00:08.4 → 00:10.0]  Unidos.
[00:10.0 → 00:15.0]  Provocando el cierre temporal de uno de los principales sistemas de distribución de
[00:15.0 → 00:22.0]  combustible del país, el ataque comenzó con credenciales de acceso robadas a través
[00:22.0 → 00:26.3]  de una cuenta VPN sin autenticación.
[00:26.3 → 00:33.6]  De doble factor, en menos de dos horas, los atacantes cifraron más de 100 GB de datos y
[00:33.6 → 00:40.4]  exigieron un rescate de 4,4 millones de dólares en Bitcoin.
[00:40.4 → 00:45.9]  Este incidente marcó un punto de inflexión en la percepción del ransomware como amenaza
[00:45.9 → 00:49.7]  a infraestructuras críticas nacionales.

TEXTO COMPLETO (95 palabras):
En mayo de 2021, el grupo de ransomware darksite atacó el oleoducto colonial pipeline en Estados Unidos. Provocando el cierre temporal de uno de los principales si

## Comparar modelos: `tiny` vs `small`

Transcribimos el mismo audio con dos modelos distintos y comparamos calidad y velocidad. Esto ilustra el trade-off fundamental: más parámetros = mejor resultado, pero más tiempo.

In [6]:
# ─── COMPARATIVA tiny vs small ────────────────────────────────────────────────
import pandas as pd

resultados = []

for nombre_modelo in ['tiny', 'small']:
    m = WhisperModel(nombre_modelo, device='cpu', compute_type='int8')
    t0 = time.time()
    segs, inf = m.transcribe(str(AUDIO_PATH), language='es')
    texto = ' '.join(s.text.strip() for s in segs)
    elapsed = time.time() - t0
    resultados.append({
        'modelo': nombre_modelo,
        'tiempo_s': round(elapsed, 1),
        'factor_velocidad': f'{inf.duration / elapsed:.1f}×',
        'palabras': len(texto.split()),
        'texto': texto,
    })
    print(f'{nombre_modelo:8s} → {elapsed:.1f}s | {texto[:80]}...')

df_comp = pd.DataFrame(resultados)
print('\n')
print(df_comp[['modelo', 'tiempo_s', 'factor_velocidad', 'palabras']].to_string(index=False))
print('\nTexto tiny:')
print(resultados[0]['texto'])
print('\nTexto small:')
print(resultados[1]['texto'])

tiny     → 0.8s | En mayo de 2021, el grupo de Ransom Werdark Saita atacó el oleoducto colonial Pi...
small    → 2.8s | En mayo de 2021, el grupo de ransomware darksite atacó el oleoducto colonial pip...


modelo  tiempo_s factor_velocidad  palabras
  tiny       0.8            64.8×        97
 small       2.8            17.9×        95

Texto tiny:
En mayo de 2021, el grupo de Ransom Werdark Saita atacó el oleoducto colonial Pipe Line en Estados Unidos. Provocando el cierre temporal de uno de los principales sistemas de distribución de combustible del. Pais, el ataque comenzó con credenciales de acceso robadas a través de una cuenta UBPN sin autenticación. De doble factor, en menos de dos horas, los atacantes cifraron más de 100 GB de datos y exigieron un rescate de 4,4. Millones de dólares en Bitcoin, este incidente marcó un punto de inflexión en la percepción del Ransom Werdark como amenaza, infraestructuras críticas nacionales.

Texto small:
En mayo de 2021, el grupo de ransomware d

## Exportar la transcripción

Guardamos dos formatos:
- **`.txt`** con timestamps — legible por humanos, útil para revisión
- **`.json`** con todos los metadatos — útil para análisis posterior con pandas o un LLM

In [7]:
# ─── EXPORTAR TRANSCRIPCIÓN ───────────────────────────────────────────────────
import json

OUT_TXT  = Path('audio/transcripcion.txt')
OUT_JSON = Path('audio/transcripcion.json')

# Formato TXT con timestamps
lineas_txt = [
    f'[{fmt_tiempo(seg.start)} → {fmt_tiempo(seg.end)}]  {seg.text.strip()}'
    for seg in lista_segmentos
]
OUT_TXT.write_text('\n'.join(lineas_txt), encoding='utf-8')

# Formato JSON con todos los metadatos del segmento
datos_json = {
    'audio_path': str(AUDIO_PATH),
    'modelo': 'small',
    'idioma': info.language,
    'confianza_idioma': round(info.language_probability, 4),
    'duracion_audio_s': round(info.duration, 2),
    'segmentos': [
        {
            'start': round(seg.start, 2),
            'end': round(seg.end, 2),
            'text': seg.text.strip(),
            'avg_logprob': round(seg.avg_logprob, 4),  # confianza del segmento
            'no_speech_prob': round(seg.no_speech_prob, 4),  # prob. de silencio
        }
        for seg in lista_segmentos
    ],
}
OUT_JSON.write_text(json.dumps(datos_json, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Guardado: {OUT_TXT}  ({OUT_TXT.stat().st_size} bytes)')
print(f'Guardado: {OUT_JSON} ({OUT_JSON.stat().st_size} bytes)')
print('\nPrimeras líneas del TXT:')
print('\n'.join(lineas_txt[:3]))

Guardado: audio/transcripcion.txt  (804 bytes)
Guardado: audio/transcripcion.json (1933 bytes)

Primeras líneas del TXT:
[00:00.0 → 00:08.4]  En mayo de 2021, el grupo de ransomware darksite atacó el oleoducto colonial pipeline en Estados
[00:08.4 → 00:10.0]  Unidos.
[00:10.0 → 00:15.0]  Provocando el cierre temporal de uno de los principales sistemas de distribución de


## BONUS — Resumir la transcripción con Ollama

Ahora que tenemos el texto, podemos pasárselo a cualquier LLM de Ollama para que lo procese. Aquí pedimos un resumen de dos líneas — pero podríamos pedir traducción, extracción de entidades, clasificación, etc.

Este es el puente entre el bloque de audio (Whisper) y todo lo que hemos visto en bloques anteriores (Ollama + CrewAI).

In [8]:
# ─── BONUS: LLM sobre la transcripción ───────────────────────────────────────
# Pasamos el texto transcrito a qwen2.5:14b y pedimos un resumen estructurado.
# Requiere que Ollama esté corriendo: `ollama serve`

import ollama

PROMPT = (
    f'El siguiente texto es la transcripción de un audio sobre un incidente de ciberseguridad.\n\n'
    f'TRANSCRIPCIÓN:\n{transcripcion_completa}\n\n'
    f'Extrae en formato de lista:\n'
    f'1. Grupo atacante\n'
    f'2. Objetivo del ataque\n'
    f'3. Vector de entrada\n'
    f'4. Impacto\n'
    f'5. Rescate exigido'
)

respuesta = ollama.chat(
    model='qwen2.5:14b',
    messages=[{'role': 'user', 'content': PROMPT}],
)

print('ANÁLISIS LLM (qwen2.5:14b):')
print('─' * 50)
print(respuesta['message']['content'])

ANÁLISIS LLM (qwen2.5:14b):
──────────────────────────────────────────────────
Basado en la transcripción proporcionada, aquí está la información extraída en el formato solicitado:

1. **Grupo atacante**: Darksite (grupo de ransomware)
2. **Objetivo del ataque**: Colonial Pipeline (uno de los principales sistemas de distribución de combustible de Estados Unidos)
3. **Vector de entrada**: Credenciales de acceso robadas a través de una cuenta VPN sin autenticación de doble factor
4. **Impacto**: Cierre temporal del oleoducto y cifrado de más de 100 GB de datos en menos de dos horas
5. **Rescate exigido**: 4,4 millones de dólares en Bitcoin
